# 01 · Tiny ImageNet: learning and pipeline verification

**Purpose.** Test whether the verified FMM-JEPA machinery trains and evaluates reproducibly on real images, and make a useful-learning experiment possible. This notebook separates engineering correctness, improvement over each encoder's own initialization, and architectural advantage.

**Inputs:** pinned `thethinkmachine/tiny-imagenet` data, the exact Notebook 00 handoff, and `configs/tiny_imagenet_smoke.yaml` by default. **Outputs:** a unique `artifacts/notebook_01/<run_id>/` with protocol, measured results, figures, certificates and handoff. Authoritative training/probe checkpoints live in `checkpoints/<run_id>-*/`, with immutable initial controls inside the run artifact.

**Expected runtime:** cached-data CUDA smoke roughly 5–15 minutes; first download/audit adds minutes. Pilot is explicitly selected and may take hours; it is not launched by default. Before either paired run, the short real-data exact-resume certificate must pass. Checkpoints save at stage/epoch boundaries and no later than 250 updates or 15 minutes; latest two valid plus best are retained.

**Exact resume:** set `FMM_JEPA_01_PROFILE` to the same profile and `FMM_JEPA_01_RESUME` to the saved run ID, then run from the bootstrap cell below in a fresh `ahr-go` kernel. Resolved configuration, registered protocol, source hashes, dataset membership and checkpoint hashes must agree. An invalid manifest raises; it never silently restarts.

The roadmap remains 00 core → **01 real-data debug** → 02 Food-101 → 03 COCO → 04 ImageNet-1K → 05 report. This task creates and executes only 01. Labels may stratify the pools and fit frozen probes; they never enter encoder pretraining. No later dataset is downloaded.

The workspace's notebook-only agreement takes precedence over the brief's suggested `.py` filenames. Reusable implementation lives in tagged `ahr-export` cells and is imported as ordinary `fmm_jepa` modules; upstream training cells are never imported or executed.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
import json, time
import nbformat
ROOT = Path(os.environ.get('FMM_JEPA_ROOT', '/home/benjamin/workspace/ahr_go/fmm_jeppa')).resolve()
_boot = nbformat.read(ROOT / 'fmm_jepa/bootstrap.ipynb', as_version=4)
_exports = {'__file__': str(ROOT / 'fmm_jepa/bootstrap.ipynb')}
for _cell in _boot.cells:
    if _cell.cell_type == 'code' and 'ahr-export' in _cell.metadata.get('tags', []):
        exec(compile(_cell.source, str(ROOT / 'fmm_jepa/bootstrap.ipynb'), 'exec'), _exports)
_exports['install_notebook_package'](ROOT)
from fmm_jepa.tiny_workflow import (resolve_config, start_run, verify_predecessor, run_environment,
    save_dataset_evidence, geometry_evidence, register_protocol, verification_suite, train_pair,
    evaluate_four, finish_run)
from fmm_jepa.training import atomic_json
from IPython.display import display, Markdown
config = resolve_config(ROOT)
run_id, artifact = start_run(ROOT, config, os.environ.get('FMM_JEPA_01_RESUME'))
print('Run:', run_id, '| profile:', config['run_mode'], '| artifacts:', artifact)

## 02 · Verify the predecessor

A past specification is not evidence. We check the actual predecessor run, its schemas, exported-definition hashes, saved clean execution, and all required certificates. We then rerun compact geometry, transfer, masking/leakage, model/backward, predictor, EMA and checkpoint tests against current code. Essential failure stops integrated training. Fresh initialization below never loads synthetic-trained weights.

In [ ]:
predecessor = verify_predecessor(ROOT, artifact)
print('Verified predecessor:', predecessor['predecessor_run_id'])
print('Fresh compact checks:', len(predecessor['fresh_certificates']), '|', predecessor['status'])

## 03 · The mathematical question

For one augmented image $x$, visible patch coordinates $C$, and target block $T_k$,
$$z_C=E_\theta(x_C;C),\quad z_{T_k}=[E_{\bar\theta}(x;\text{all positions})]_{T_k},\quad \hat z_{T_k}=P_\phi(z_C,C,T_k).$$
The student forms raw patches with an unfold operation, then gathers **only visible raw patches before learned embedding and mixing**, retaining their original coordinates. The teacher sees the complete, identical augmented image. The small conventional Transformer predictor receives visible features and target positions, never teacher latents.

Let $D$ be feature width and $B$ the valid example count. The imported objective is
$$\mathcal L=\frac1{BK}\sum_{b=1}^B\sum_{k=1}^K\frac1{|T_{bk}|D}\sum_{i\in T_{bk}}\sum_{d=1}^D h_1\!\left(\hat z_{bkid}-\operatorname{sg}[\operatorname{LN}(z_{bk i})]_d\right),$$
where $h_1(u)=u^2/2$ for $|u|<1$ and $|u|-1/2$ otherwise. Each example and target block receives equal weight; padded positions are excluded. Overlapping target blocks contribute separately. The teacher already has its encoder's final affine LayerNorm; the loss adds the inherited nonaffine target LayerNorm. No pixel reconstruction, classification, contrastive loss or collapse penalty is added.

After each **successful** optimizer step, $\bar\theta\leftarrow m\bar\theta+(1-m)\theta$. The teacher has no optimizer or gradients, stays in evaluation mode and follows explicit buffer handling. EMA and LayerNorm do not prove that collapse is impossible.

The FMM mixer adds local content-dependent attention and hierarchical far messages:
$$m_i=m_i^{near}+m_i^{far},\qquad P2M\to M2M\to M2L\to L2L\to L2P.$$
P2M/L2P and M2M/L2L use fixed tensor-product Chebyshev interpolation geometry. Value/output projections, relative-offset M2L node-to-node translation matrices shared across feature channels and the receiver gate are learned. The FMM encoder has no global-attention fallback. Normalization, residual/MLP scaffold and predictor remain shared with ViT. The inherited implementation stores 49 offset matrices, including offsets unused on this grid; actual counts include them.

The key learning comparison is **trained features minus the same encoder's saved initial features**. Beating 0.5% uniform guessing or decreasing JEPA loss alone does not answer that question.

In [ ]:
from fmm_jepa import reporting
from fmm_jepa import tiny_figures as figures
figures.setup_style()
figures.save_figure(reporting.architecture_figure(), artifact, 'architecture')
figures.save_figure(figures.pretrain_probe_diagram(), artifact, 'pretraining_and_probe')

## 04 · Environment and execution profile

CUDA smoke exercises the intended width192/depth4, patch8, p4 FMM architecture with conservative microbatch4 and accumulation2. CPU-only smoke explicitly reduces width/depth and updates. The code uses one GPU sequentially even if more are present. FP32, disabled TF32 and deterministic algorithms are recorded; mixed precision has not been authorized by a dtype certificate. Installed PyTorch/CUDA are preserved.

In [ ]:
environment = run_environment(config)
atomic_json(artifact / 'environment.json', environment)
display(Markdown(f"Device **{environment['device']}**, FP32; GPUs {environment['gpu_names']}. "
                 f"Microbatch **{environment['microbatch']}**, accumulation **{environment['accumulation_steps']}**, "
                 f"effective batch **{environment['effective_batch_size']}**. CPU threads {environment['cpu_threads']}."))
print('Libraries:', environment['packages'])
print('Free/total device bytes:', environment['cuda_free_total_bytes'])
print(environment['storage_persistence'])

## 05 · Pinned real data and inspection

The intended repository is verified directly; its copied card example does not select another mirror. The immutable source revision and converted Parquet revision are separate. Direct Parquet loading avoids executing legacy remote dataset code. The adapter checks file hashes, actual `train`/`valid` split names, schema and all200 class identities, including pinned metadata crosschecks where Parquet stripped the ClassLabel schema.

Images decode lazily for experiments, with explicit RGB conversion; a cached full-source integrity audit records native sizes, mode counts and every exclusion. Public/ungated access is recorded separately from the empty declared license and ImageNet access terms. See the dataset manifest and pinned primary references; this notebook infers no additional permissions.

In [ ]:
from fmm_jepa.tiny_imagenet import prepare_dataset, make_batch
bundle = prepare_dataset(ROOT, config)
data_summary = save_dataset_evidence(bundle, artifact)
print('Repository:', config['data']['repo_id'])
print('Source:', config['data']['source_revision'])
print('Converted Parquet:', config['data']['parquet_revision'])
print('Dataset fingerprint:', data_summary['dataset_fingerprint'])

## 06 · Separate pools before selecting subsets

Seed314159 stratifies the source training split into candidate450/class training and50/class development pools. All source `valid` examples are held out. Exact decoded-RGB duplicate groups are resolved **before subset selection** with heldout > development > training priority, then lowest source row. The source labels are never rewritten. This dataset contains27 duplicate pairs with conflicting labels; 15 pairs cross candidate partitions. Near-duplicate semantic similarity is not certified by this exact-hash audit.

The resulting pools contain89,975 training,9,998 development and10,000 held-out examples. Smoke selects10/class and5/class (2,000/1,000); pilot selects100/class and10/class (20,000/2,000). Exact IDs and hashes are saved. The probe intentionally reuses labels for the encoder's training pool; development is absent from encoder pretraining and held-out images are absent from all choices.

In [ ]:
print('Clean pools:', data_summary['pool_sizes'])
print('Selected:', data_summary['selected_sizes'])
print('Split/subset identity:', data_summary['split_sha256'])

## 07 · Native geometry and actual far interactions

64×64 inputs with patch8 produce an8×8 grid: $N_{grid}=64$. Leaves cover2×2 patches, neighbor radius is one cell, and levels allocate1/4/16 cells. Order p4 means **16 tensor-product nodes per cell per channel**. That overhead is substantial for64 tokens.

At this grid size, genuine well-separated M2L interactions occur at the leaf level; no coarser M2L signal is present. Notebook00's larger fixture certifies the deeper routing path. The following evidence counts real geometry and occupancy for a visible context, rather than equating an allocated cell with an occupied one.

In [ ]:
example_ids = [bundle['train_ids'][i] for i in (0, 10, 20, 30)]
example_batch = make_batch(bundle['dataset'], example_ids, [0]*4, config)
geometry = geometry_evidence(config, example_batch, artifact)
for key in ('n_grid','order','nodes_per_cell','allocated_cells_per_level','occupied_cells_per_level_first_context',
            'fullgrid_near_ordered_pairs','far_cell_interactions_per_level','visible_counts','target_counts'):
    print(key, ':', geometry[key])
figures.save_figure(reporting.query_figure(grid=(8,8), query=27, dim=8), artifact, 'query_near_far')

## 08 · Augmentation, masks and information isolation

A moderate random resized crop uses area0.6–1.0, aspect0.75–4/3, then horizontal flip probability0.5 and fixed channel mean/std0.5. Evaluation uses the complete native RGB image and fixed normalization. No development-derived pixel statistics are fitted.

Four target blocks request15–20% area, aspect0.75–1.5; context requests85–100% then removes the target union. Minimum visible count4, at most32 rectangle proposals and16 whole-mask retries, and a recorded strict fallback handle integer rounding without overlap. The inherited geometry and disjointness semantics are preserved. Mask, augmentation and order streams derive from separate seeds, stable sample IDs and consumed occurrences; model execution cannot alter them. Real integer shapes, realized areas and fallback flags are retained in mask metadata.

In [ ]:
figures.save_figure(figures.examples_figure(example_batch, config), artifact, 'real_images_masks')

## 09 · Controlled model construction

Both encoders use the same input resolution, width, depth, coordinates, residual/MLP scaffold, predictor, loss and EMA rule. ViT uses optimized SDPA; actual dispatch is profiled. FMM replaces the encoder mixer only. Equal width/depth does not equal parameter count, so this is a **same-scaffold** comparison. Models are initialized independently with sensible shared nonmixer components copied without tensor aliasing.

In [ ]:
from fmm_jepa.tiny_training import initialize_paired_models
models, parameter_counts, initialization = initialize_paired_models(config, artifact)
for kind, counts in parameter_counts.items():
    print(kind, counts)

## 10 · Register the comparison before collecting outcomes

AdamW uses lr3e−4, weight decay0.05, bias/one-dimensional normalization exclusions, warmup5% and cosine decay. Schedules advance on successful updates only; the last scheduled learning rate is zero. EMA momentum progresses0.996→1.0; clipping norm1. Fixed full microbatches accumulate with divided losses; partial accumulation is rejected. Rejected nonfinite attempts consume recorded inputs but do not advance EMA/schedules; excessive failures stop. Any asymmetric data consumption invalidates the equal-data comparison.

The primary encoder is the **final-budget EMA target**, never each model's best JEPA loss checkpoint. The protocol below records seeds, dataset/source identities, architecture/counts, hardware, precision, probe budget, stop rules and exported source hashes. Changes require a new run ID. Smoke is always learning-inconclusive. A later pilot requires a positive FMM-minus-initial top1 paired interval and no unresolved degeneracy to satisfy this registered readiness criterion.

In [ ]:
protocol = register_protocol(ROOT, artifact, run_id, config, bundle, parameter_counts, initialization, environment, predecessor)
print('Registered protocol:', artifact / 'protocol.json')

## 11 · Preserve own-initialization controls

Before any training, both exact initial EMA encoders and full initial model states are saved and hash checked. These immutable controls are distinct from resumable training checkpoints. Every trained-versus-initial comparison uses the same architecture's own state and the same frozen-probe protocol.

In [ ]:
for kind, states in initialization['states'].items():
    print(kind, 'initial teacher:', states['teacher']['path'], '| SHA256:', states['teacher']['sha256'])
print('Independent storage verified:', initialization['independent_storage_verified'])

## 12 · Verify real-image integration before training

Required evidence includes context/target disjointness, six predictor inputs with positions rather than teacher latents, unchanged loss after label shuffling, unchanged context after hidden-pixel perturbation, finite forward/backward, live near/far gradients, teacher isolation and numerical EMA. Padding/dropout checks inherit the freshly verified core. Accumulation is compared against a full batch.

The controlled resume test runs each architecture for4 updates versus2+save/reload+2, on real images in CPU FP32 with one deterministic thread and zero workers. It compares consumed sample IDs, augmentations/masks, losses, all model/optimizer/scheduler/RNG state, and refuses corrupt checkpoints. This is the short certificate required before any long pilot. Evaluation tests additionally cover train-only normalization, cache rejection, frozen state and exact probe resume.

In [ ]:
certificates = verification_suite(ROOT, artifact, config, bundle, models)
print('Real integration, accounting, training resume and frozen-probe verification completed.')

## 13 · Execute the bounded paired training budget

The default smoke processes20 successful updates per model, effective batch8:160 image exposures each, drawn from the selected2,000-image pool. This is much less than an epoch and cannot test representation quality convincingly. Both architectures receive identical consumed IDs, occurrences, augmentations and masks. The reusable runner trains sequentially and resumes automatically only from validated state.

Fixed raw representation diagnostics use one development image per class at steps0/10/20, outside optimizer training and inside a preserved RNG context. Diagnostics must not change the next training input.

In [ ]:
trained, diagnostics = train_pair(ROOT, artifact, run_id, config, bundle, models, protocol)
print('Actual completed budgets:', {kind: value['resume'] for kind, value in trained.items()})

## 14 · The unexecuted pilot uses the same path

`configs/tiny_imagenet_pilot.yaml` explicitly requests5,000 successful updates/model, microbatch8×accumulation16=effective128,20,000 training/2,000 development examples and30 probe epochs. Set `FMM_JEPA_01_PROFILE=pilot` before fresh-kernel execution to select it. `full_pool` uses all duplicate-cleaned training/development members and is also opt-in. Neither mode is automatically launched by available hardware. Only `evaluation.heldout_enabled=true` in an explicitly selected pilot permits held-out prediction after all four choices are fixed.

A run stopped early is incomplete, with its actual counters and valid resume state retained. No pilot outcome is inferred from smoke success.

## 15 · Inspect training and representations

Gradient norms are reported before clipping and for near/far branches after clipping. A falling loss can coexist with poor features. Across-image pooled-feature standard deviation, norm and cosine are therefore accompanied by covariance eigenvalues and
$$r_{eff}=\exp(-\sum_j q_j\log q_j),\quad q_j=\lambda_j/\sum\lambda.$$
Zero total variance gives rank0 explicitly; the ceiling is $\min(D,n-1)$. Diagnostics are measured **before** optional probe standardization. Matching-position token variation compares different images at the same patch position; it does not confuse positional variation within an image with image-specific content. Rank<2 and variance≤1e−12 are advisory inspection flags, not universal theorems. Nonzero far outputs/gradients demonstrate activity, not semantics.

In [ ]:
training_rows = [row for value in trained.values() for row in value['metrics']]
figures.save_figure(figures.training_figure(training_rows, config['run_mode']), artifact, 'training_traces')
figures.save_figure(figures.diagnostics_figure(diagnostics, config['run_mode']), artifact, 'representation_diagnostics')
for row in diagnostics:
    print(row['model_id'], row['step'], 'std', round(row['feature_std_mean'], 5),
          'rank', round(row['effective_rank'], 3), 'of', row['effective_rank_ceiling'])

## 16 · Checkpoint/resume evidence and its scope

The real-data certificate already passed before training. Checkpoints include student, teacher, predictor, optimizer, schedule/scaler policy, all RNGs, consumed order, successful/attempt counters, configuration/protocol/dataset/source identities and metrics. One writer owns each run. Atomic manifests identify the latest valid boundary and retain latest two plus best; best-loss retention is for recovery, not model selection.

Exact equivalence is certified for the pinned single-thread CPU FP32 setting. A preliminary two-thread CPU FMM replay produced identical losses with state differences up to3.28e−7; nondeterministic CUDA repeats also differed slightly. These observations are preserved in development verification. The resolved deterministic GPU setting passes exact hidden-pixel isolation; no cross-device/version bitwise promise is made.

In [ ]:
print('Controlled resume evidence:', artifact / 'resume/resume_certificate.json')
for kind, value in trained.items():
    print(kind, 'authoritative checkpoint:', value['checkpoint_dir'], '|', value['resume'])

## 17 · Frozen complete-image feature extraction

The registered image representation is $z(x)=64^{-1}\sum_i E_{\bar\theta}(x)_i$: complete-image EMA patch outputs **after the encoder's final affine LayerNorm**, averaged over valid patches. The predictor and the additional loss-only nonaffine normalization are absent. Encoders stay frozen in evaluation mode.

Feature caches bind dataset revisions, exact sample IDs, encoder state hash/kind, deterministic preprocessing, pooling and normalization. Stale or corrupt caches are refused. Training-feature mean/std are fit once per representation and applied unchanged to development and any explicitly permitted held-out set.

## 18 · Four equally budgeted linear probes

Only $W,b$ in $Wz(x)+b$ learn from labels. All four controls use identical classifier initialization, data order, AdamW lr0.01/weightdecay0.0001, batch256 and the same epoch budget (2 smoke,30 pilot). Development top1 selects the best epoch, earliest on ties; every control still completes its full budget. The encoder state hash is checked before/after. Both probe and feature extraction resume safely. Per-example predictions preserve stable IDs for paired comparisons.

In [ ]:
probes, comparison, evaluation_performance = evaluate_four(ROOT, artifact, run_id, config, bundle, trained, initialization)
figures.save_figure(figures.probe_figure(probes, config['run_mode']), artifact, 'four_probe_controls')

## 19 · Measure comparable workloads

Dedicated encoder timings use the same full64-token real-image batch, precision and model scaffold, with warmup and CUDA synchronization around measurement. They are separate from data/preprocessing, complete JEPA forward/backward/optimizer/EMA steps, feature extraction and probe fitting. Normal training CUDA event timing is flushed at checkpoint boundaries. Peak allocations record live baseline and incremental memory where available, and models are measured sequentially.

Inspect actual SDPA events rather than assuming FlashAttention or quadratic materialized attention memory. FMM's predictor still uses attention. A64-token measurement cannot establish asymptotic complexity or high-resolution learning quality.

In [ ]:
from fmm_jepa.tiny_performance import measure_encoder_performance
encoder_performance = measure_encoder_performance({k:v['model'] for k,v in trained.items()}, example_batch, config,
    checkpoint_ids={k:v['checkpoint_manifest']['latest'] for k,v in trained.items()})
performance = [v['performance'] for v in trained.values()] + encoder_performance + evaluation_performance
figures.save_figure(figures.performance_figure(performance, config['run_mode']), artifact, 'measured_performance')
for row in encoder_performance:
    print(row['model_id'], row)

## 20 · Paired contrasts and uncertainty

Report three distinct quantities: $\Delta_{ViT}=a_{ViT,trained}-a_{ViT,initial}$, $\Delta_{FMM}=a_{FMM,trained}-a_{FMM,initial}$, and $\Delta_{arch}=a_{FMM,trained}-a_{ViT,trained}$. Paired bootstrap intervals resample the same development-image IDs. They quantify evaluation-image uncertainty conditional on these fixed models; they exclude training-seed variation. One paired seed and a short smoke cannot establish an architectural advantage.

In [ ]:
figures.save_figure(figures.comparison_figure(comparison), artifact, 'paired_comparisons')
for name, values in comparison['deltas'].items():
    d = values['correct_top1']
    print(name, f"{d['delta_percentage_points']:+.2f} percentage points; 95% CI {d['ci95_percentage_points']}")

## 21 · Keep engineering and learning ledgers separate

E1–E14 cover core compatibility, real data/splits/masks, information isolation, gradients/updates/EMA, accounting/resume, frozen controls, performance and clean execution. L1–L6 separately record pilot completion, initialization contrasts, collapse inspection, held-out status, seed count and limitations. Software success never depends on FMM winning.

The final external fresh-kernel executor closes E14 only after this notebook actually finishes without an error. All artifacts below remain auditable; no skipped measurement gets a fabricated row.

In [ ]:
handoff = finish_run(ROOT, artifact, run_id, config, predecessor, bundle, parameter_counts, protocol,
    certificates, trained, diagnostics, comparison, performance)
print('Report:', artifact / 'engineering_report.md')
print('Ledger:', artifact / 'certificate_ledger.json')
print('Learning evidence:', handoff['learning_evidence'])
print('Held-out predictive evaluation:', handoff['heldout_status'])

## 22 · Handoff and bounded next decision

The machine-readable handoff records distinct `engineering_ready`, `learning_evidence`, `ready_for_notebook_02`, `readiness_reason` and `readiness_override` fields. A smoke-only run always leaves learning inconclusive and Food-101 readiness false. The next experiment is the explicitly selected Tiny ImageNet pilot, not an automatic dataset escalation.

If a later authorized pilot establishes useful FMM learning and closes essential engineering gates, the bounded Food-101 comparison should start both architectures fresh, with all four controls and separately registered budgets. Tiny ImageNet weights are not implicitly transferred. No Notebook02 is built here.

See `docs/notebook_02_handoff.md`, the run's `engineering_report.md`, `comparison_summary.json`, and `handoff_01_to_02.json` for actual results. Source references and declared implementation differences are in `docs/notebook_01_references.md` and `docs/notebook_01_protocol.md`.

### Executed smoke readout

External fresh-kernel execution passed for run `20260923T173558Z-2cc8d346` in approximately64seconds with the prepared dataset cache. **All14 engineering gates passed.** Both models completed20successful/20attempted updates,160image exposures each; paired consumed sample/augmentation/mask traces match. Held-out predictive evaluation remains **NOT_RUN**.

|Representation|Development top1|Development top5|
|---|---:|---:|
|ViT initialization|2.9%|8.2%|
|ViT trained|2.8%|8.2%|
|FMM initialization|2.3%|10.0%|
|FMM trained|2.2%|9.9%|

Neither encoder improved over its own initialization in this short run. FMM's top1 difference is −0.1percentage points, paired95%interval[−0.5,+0.2]. These tiny smoke changes do not establish benefit or lasting harm. Mean raw feature standard deviations remain nonzero (ViT0.210,FMM0.188), but effective ranks are only2.54/2.52 against a192ceiling, already similarly low at initialization. No constant-output collapse was observed; the strongly concentrated covariance needs inspection during a completed pilot. Passing the advisory rank<2 threshold is not evidence of healthy or semantic representations.

FMM's far messages and gradients are active. That is an implementation result. FMM used more trainable parameters (2,606,088versus2,263,296), and was slower here: median complete updates273msversus152ms; same full-grid batch4 encoder forwards23.9msversus4.44ms. This establishes no quality–compute advantage.

**Engineering ready: true. Learning evidence: inconclusive. Ready for Food-101: false.** The next unexecuted experiment is the explicitly selected Tiny ImageNet pilot. The initial failed integration run and its inference-cache repair/regression certificate are preserved; it is not counted as a completed comparison.
